In [ ]:
%load_ext autoreload
%autoreload 2

import xarray as xr 
import tams
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import pandas as pd
import seaborn as sns
import os
import numpy as np
import analysis_functions as af2

from joblib import Parallel, delayed
from scipy.stats import gaussian_kde


import warnings
warnings.filterwarnings('ignore')

* Extracted from tams_predata.ipynb

* In order to take care of some nan values found in ctt, we're going to crop the lon coordinates as they are the main problem (see nan_check for more information)

In [ ]:
root = "./"
pr_pre = "precipitation_IMERG_1999_WA" # This can be changed to match the data and resolution.
data_path_pr = f"{root}TFM/{pr_pre}_0.25x0.25.nc"

temp_pre = "tb_merg_1999_WA" # This can be changed to match the data and resolution.
data_path_temp = f"{root}TFM/{temp_pre}_0.25x0.25.nc"

CHUNK_T = 96   # tune: bigger → more RAM, fewer tasks; 96 ≈ 2 days of 30-min steps

pr_data = (
    xr.open_dataset(data_path_pr,   chunks={"time": CHUNK_T, "lat": -1, "lon": -1})
      .transpose("time", "lat", "lon")
      .rename({"precipitation": "pr"})
)
temp_data = (
    xr.open_dataset(data_path_temp, chunks={"time": CHUNK_T, "lat": -1, "lon": -1})
      .rename({"Tb": "ctt"})
)

tbI = temp_data["ctt"].dropna(dim='time', how='all')
tp  = pr_data["pr"]

# Interpolate in order to make merge faster for different resolutions
#pr_data = pr_data.interp_like(temp_data, method='nearest')

- Let's fix the exact timestamp where we have a row full of nans. I'll skip them, returning an empty GeoDataFrame for that moment. 

In [ ]:
ds = xr.Dataset({'tb': tbI, 'tp': tp})

cesI, _ = tams.identify(tbI.compute(), parallel=True);

cesI_all = af2.Parallel_nan_ds(cesI, ds)

In [ ]:
cesI_c = cesI_all.copy()
print(len(cesI_all), len(cesI_c))
print(True if cesI_all == cesI_c else False)

- Do tracking

In [ ]:
proj = -10
times = tbI.time.values.tolist()

ces_T = tams.track(cesI, times, u_projection=-10) #Change cesI to cesI_c to test

In [ ]:
ce_clasI = tams.classify(ces_T)

## Other Year

In [ ]:
root = "./"
pr_pre = "precipitation_IMERG_2024_WA" # This can be changed to match the data and resolution.
data_path_pr = f"{root}TFM/{pr_pre}_0.25x0.25.nc"

temp_pre = "tb_merg_2024_WA" # This can be changed to match the data and resolution.
data_path_temp = f"{root}TFM/{temp_pre}_0.25x0.25.nc"

CHUNK_T = 96 

pr_data = (
    xr.open_dataset(data_path_pr,   chunks={"time": CHUNK_T, "lat": -1, "lon": -1})
      .transpose("time", "lat", "lon")
      .rename({"precipitation": "pr"})
)
temp_data = (
    xr.open_dataset(data_path_temp, chunks={"time": CHUNK_T, "lat": -1, "lon": -1})
      .rename({"Tb": "ctt"})
)
tbI = temp_data["ctt"]#.dropna(dim='time', how='all')
tp  = pr_data["pr"]


In [ ]:
ds = xr.Dataset({'tb': tbI, 'tp': tp})
timeI = tbI.time.values.tolist()

cesI, _ = tams.identify(tbI.compute(), parallel=True);




In [ ]:
cesI_all = af2.Parallel_ds(cesI, ds)

In [ ]:
# Añado variables de interés a los ces identificados:
def fun(ds,ce):
    if not ce.empty:
        ce = tams.data_in_contours(ds.tb, ce, agg=("mean", "min","max"), merge=True)
        ce = tams.data_in_contours(ds.tp, ce, agg=("mean","min", "max"), merge=True)
    return ce

cesI_all = Parallel(n_jobs=-2, verbose=10)(
    delayed(fun)(ds.isel(time=i).copy(deep=False),ce.copy())
    for i, ce in enumerate(cesI)
)


In [ ]:
cesI_c = cesI_all.copy()
print(type(cesI_c))

In [ ]:
af2.save_parquet(cesI_c, "tams_2024_025")

In [ ]:
ces_par = af2.list_from_parquet("./parquet_sets/tams_2024_025.parquet", len(tbI.time))
proj = -10
times = tbI.time.values.tolist()



- We compare two methods, they should be the same and work perfectly.

In [ ]:
ces_T = tams.track(ces_par, times, u_projection=-10)

In [ ]:
ces_t = tams.track(cesI_c, times, u_projection=-10)

In [ ]:
cesI_c.groupby(["mcs_id", "mcs_class"], observed=True)

- Idem.

In [ ]:
ce_clasI = tams.classify(ces_T)

In [ ]:
ce_clas_i = tams.classify(ces_t)

In [ ]:
af2.dibujoMCS(ce_clasI)

In [ ]:
ce_clas_i

In [ ]:
def _add_spatial_temporal_cols(ce: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Derive cent_lat, cent_lon, and day_yr from geometry and time.
    Returns a copy with the new columns added.
    """
    ce = ce.copy()
    centroids = ce.geometry.to_crs(epsg=3857).centroid.to_crs(epsg=4326)
    ce["cent_lat"] = centroids.y
    ce["cent_lon"] = centroids.x
    ce["day_yr"]   = ce["time"].dt.dayofyear
    return ce

ce_ex = _add_spatial_temporal_cols(ce_clasI[ce_clasI["mcs_class"] == "MCC"])
ce_ex

- BUILD A FUNCTION TO TEST THEN PASS IT TO LIBRARY.

In [ ]:
_MCS_ORDER = ["DSL", "DLL", "CCC", "MCC"] # This follows Nunez Ocasio Classification
_KDE_KWARGS = dict(fill=False, alpha=0.6, common_norm=True)
_SAHEL_EXTENT = (lonmin, lonmax, latmin, latmax) = (-40,50,0,40) #(°W, °E, °S, °N)

def _kde_scatter(ax, data:pd.DataFrame, x: str, y:str, clip=None):
    """Auxiliar funciton - Scatter + KDE overlay on a given axis."""

    ax.plot(data[x], data[y], ".", ms=1, mec="none", mfc="0.35", alpha=0.9)
    sns.kdeplot(
        x=x, y=y,
        color= sns.color_palette()[0],
        clip=clip,
        ax=ax,
        data=data,
        **_KDE_KWARGS,
    )

def seasonal_latlon(
        tbI: xr.DataArray,
        ce_clasI: gpd.GeoDataFrame,
        mcs_class: str,
) -> plt.Figure:
    """
    Plot the seasonal (day-of-year) relationship between MCS location for a
    given MCS class, across three panels:
 
      1. Day-of-year vs latitude
      2. Longitude vs day-of-year
      3. Map: longitude vs latitude with coastlines
 
    The map extent is inferred automatically from the data, with an optional
    padding applied on each side.
 
    Parameters
    ----------
    tbI : xr.DataArray
        Brightness temperature array with a 'time' coordinate in datetime
        format. Used only to derive the season boundaries (first/last day
        of year).
    ce_clasI : gpd.GeoDataFrame
        GeoDataFrame produced by ``tams.classify()``. Must contain columns:
        'mcs_class', 'cent_lat', 'cent_lon', 'day_yr', 'mcs_id', 'itime'.
    mcs_class : str
        MCS class to plot. Must be one of 'DSL', 'DLL', 'CCC', 'MCC'.
 
    Returns
    -------
    matplotlib.figure.Figure
    """

    if mcs_class not in _MCS_ORDER:
        raise ValueError(f"mcs_class must be one of {_MCS_ORDER}, got {mcs_class!r}")
    
    start = int(tbI.time.min().dt.dayofyear)
    end = int(tbI.time.max().dt.dayofyear)

    ce = _add_spatial_temporal_cols(ce_clasI[ce_clasI["mcs_class"] == mcs_class])

    if ce.empty:
        raise ValueError(f"No data found for mcs_class={mcs_class!r}.")
    
    gb = ce.groupby(['mcs_id', 'itime'])
    stats = pd.DataFrame({
        "day_yr": gb.day_yr.mean().astype(float),
        "cent_lat": gb.cent_lat.mean().astype(float),
        "cent_lon": gb.cent_lon.mean().astype(float) + 360,
    })

    lonmin, lonmax, latmin, latmax = _SAHEL_EXTENT
 
    # ── Figure ───────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(10, 10))
    fig.suptitle(f"Seasonal lat/lon distribution — {mcs_class}", y=1.01)
 
    # Panel 1: day-of-year vs latitude
    ax1 = fig.add_subplot(2, 2, 1)
    _kde_scatter(ax1, stats, x="day_yr", y="cent_lat", clip=(0, None))
    ylim = ax1.get_ylim()
    ax1.vlines([start, end], *ylim, colors="orange", label="Season bounds")
    ax1.set_ylim(ylim)
    ax1.set_xlabel("Day of year")
    ax1.set_ylabel("Mean latitude [°]")
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)
 
    # Panel 2: longitude vs day-of-year
    ax2 = fig.add_subplot(2, 2, 2)
    _kde_scatter(ax2, stats, x="cent_lon", y="day_yr", clip=(0, None))
    xlim = ax2.get_xlim()
    ax2.hlines([start, end], *xlim, colors="orange", label="Season bounds")
    ax2.set_xlim(xlim)
    ax2.set_xlabel("Mean longitude [°]")
    ax2.set_ylabel("Day of year")
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)
 
    # Panel 3: map
    ax3 = fig.add_subplot(2, 1, 2, projection=ccrs.PlateCarree())
    stats_map = stats.copy()
    stats_map["cent_lon"] -= 360  # restore -180/180 for PlateCarree
 
    _kde_scatter(ax3, stats_map, x="cent_lon", y="cent_lat")
    ax3.coastlines(linewidth=2)
    ax3.set_extent([lonmin, lonmax, latmin, latmax], crs=ccrs.PlateCarree())
    ax3.set_xlabel("Longitude [°]")
    ax3.set_ylabel("Latitude [°]")
 
    fig.tight_layout()
    return fig

seasonal_latlon(tbI, ce_clasI, "MCC")

In [ ]:
print(len(ce_clasI))                  # number of CE rows
print(ce_clasI.mcs_id.nunique()) 